# CropFusion — Export release (end-to-end)

Kaggle notebook for the Training Platform — bootstraps the environment, then
exports the trained checkpoint to TorchScript + ONNX via
`export_release.py` (checkpoint is mounted from the private
`cropfusion-checkpoints` dataset uploaded by the orchestrator). The release
manifest + model files land under `training/artifacts/releases`.

- Dataset (attach this notebook to it): `shathanandabhatn/crop-yield-forecasting-karnataka-dakshina-kannada`

In [ ]:
import os
import sys
from pathlib import Path

REPO_ROOT = Path('/kaggle/working/CropPrep')
sys.path.insert(0, str(REPO_ROOT))
os.chdir(REPO_ROOT)
print(f'repo root: {REPO_ROOT}')


## 1.1 P100 GPU fix

Kaggle's base PyTorch (cu128) dropped Pascal `sm_60` kernels, so the P100
cannot execute any kernel (`CUDA error: no kernel image is available`).
Reinstall torch from the cu126 index, which still ships `sm_60` cubins
(verified fix, kaggle/docker-python#1546).


In [ ]:
import subprocess, sys

subprocess.run(
    [sys.executable, '-m', 'pip', 'uninstall', '-y', 'torch', 'torchvision', 'torchaudio'],
    check=False,
)
subprocess.run(
    [sys.executable, '-m', 'pip', 'install', '--no-cache-dir',
     'torch==2.10.0', 'torchvision==0.25.0', 'torchaudio==2.10.0',
     '--index-url', 'https://download.pytorch.org/whl/cu126'],
    check=True,
)
import torch
print('torch', torch.__version__,
      '| cuda', torch.cuda.is_available(),
      '| device', torch.cuda.get_device_name(0) if torch.cuda.is_available() else None,
      '| arch', torch.cuda.get_arch_list())


## 1. Bootstrap

Same environment + data-source setup as the training notebook.

In [ ]:
!python training/kaggle/scripts/bootstrap.py --skip-install

## 2. Exporter + release readiness

Loads the model configuration, initialises the workspace and reports the
exporter descriptor + release output paths.

In [ ]:
import inspect
from training.kaggle.config import load_paths_config, WorkspaceLayout
from training.kaggle.workspace import WorkspaceManager
from training.models.exporter import ModelExporter
from training.models.factory import ModelFactory

paths = load_paths_config()
layout = WorkspaceLayout.resolve(paths, repo_root=REPO_ROOT)
workspace = WorkspaceManager(layout)
workspace.create()

print('workspace outputs:', workspace.layout.outputs)
print('checkpoint dir   :', workspace.layout.checkpoints)
print('exporter         :', inspect.signature(ModelExporter.__init__))
print('factory          :', ModelFactory.__module__ + '.' + ModelFactory.__name__)

## 3. Export release

Exports the checkpoint mounted from the private `cropfusion-checkpoints`
dataset (uploaded by the orchestrator after training) into
`training/artifacts/releases` (TorchScript + ONNX + model config + manifest).

In [ ]:
import os
from pathlib import Path

ckpt = Path('/kaggle/input/cropfusion-checkpoints/checkpoint.pt')
print('mounted checkpoint:', ckpt)
if not ckpt.exists():
    raise SystemExit('checkpoint dataset not mounted: ' + str(ckpt))
print('checkpoint size  :', ckpt.stat().st_size, 'bytes')

In [ ]:
!python training/kaggle/scripts/export_release.py \
    --checkpoint /kaggle/input/cropfusion-checkpoints/checkpoint.pt \
    --output /kaggle/working/CropPrep/training/artifacts/releases; echo "export_exit=$?"

## 4. System check

Confirms the environment still validates after the export phase.

In [ ]:
!python training/kaggle/scripts/system_check.py